# Lesson 24 Lab — Benchmark Design: Throughput, Latency, Concurrency, and Memory

**Puzzle:** How can the same GPU path improve throughput while worsening latency?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Throughput, latency, concurrency, and memory are coupled but not interchangeable. A larger batch can raise examples per second while increasing per-request waiting time and peak memory. A useful benchmark starts from an SLO and a request distribution, then reports enough axes to explain why one configuration wins.


## 0. Predict before running

1. Predict how operator latency, examples per second, and peak allocation change from batch 1 to 128.
2. Explain why this operator benchmark cannot report time to first token or queueing latency.
3. Choose percentile and load information required for a serving comparison.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Benchmark outputs include latency distribution, throughput, concurrency, queueing, TTFT, token latency, memory, power/cost, and workload shape. They cannot be collapsed into one number.

- Latency is per request; throughput is completed work per unit time.
- Batching amortizes overhead but increases queueing and memory demand.
- Median alone hides tail behavior; warm-up and repeated samples must be recorded.


## 2. Derive the mechanism

For a fixed operator, throughput is `batch / latency`; batching can raise throughput while each item waits longer. In a service, arrival rate and queueing add latency beyond GPU execution.

For a synchronous batch B with measured operator time T, idealized throughput is `B/T`. That calculation excludes arrivals, batching delay, scheduler overhead, token-by-token Decode, and response streaming. In a service, increasing concurrency can improve GPU utilization until queueing and memory pressure drive tail latency or rejection.

Latency distributions also matter: median describes a typical warm request, while p95/p99 expose interference and queueing. Peak CUDA allocation is not total process memory and should be paired with reserved memory and cache capacity when deployment fit is evaluated.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "24-benchmark-design"
device = require_cuda()
torch.manual_seed(2026 + 24)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | batch-1 BF16 two-layer MLP operator workload |
| Candidate | the same operator at batches 8, 32, and 128 |
| Held constant | model, hidden sizes, dtype, GPU, warm-up five, repeats twenty |
| Measurements | median/p90 operator latency, derived examples/s, peak allocated MiB |
| Evidence | `pytorch-gpu` |

**Experiment:** Benchmark a CUDA MLP over several batch sizes, recording median, p90, examples per second, and peak allocated memory.


## 5. Read the experiment code

The lab sweeps batch size and reports median, p90, examples/s, and peak allocated memory; it labels the result as an operator workload, not a server test.

The notebook constructs one fixed BF16 MLP, allocates each batch input, resets peak-memory statistics, and records twenty CUDA-event samples after warm-up. Throughput is derived from batch divided by median device time, making its simplified assumptions explicit.

No request scheduler, tokenizer, KV cache, network, or output loop is present. The evidence is an operator batching curve that teaches metric relationships, not an online service benchmark.

Only after these variables match the protocol should the cell be executed.


In [2]:
model=torch.nn.Sequential(torch.nn.Linear(2048,4096,bias=False),torch.nn.GELU(),torch.nn.Linear(4096,2048,bias=False)).to(device).bfloat16(); rows=[]
for batch in (1,8,32,128):
    x=torch.randn(batch,2048,device=device,dtype=torch.bfloat16); torch.cuda.reset_peak_memory_stats(); timing=cuda_benchmark(lambda:model(x),warmup=5,repeats=20)
    rows.append({"batch":batch,"timing":timing,"examples_per_second":round(batch/(timing["median_ms"]/1000),2),
                 "peak_allocated_mib":round(torch.cuda.max_memory_allocated()/2**20,3)})
result=base_result(24,"pytorch-gpu"); result.update({"operator_workload":rows,
    "conclusion":"Batching changed throughput, latency, and memory in different directions; no service queueing was modeled."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Batch 1 median | 0.046176 ms |
| Batch 1 throughput | 21,656.27 examples/s |
| Batch 32 median | 0.045968 ms |
| Batch 32 throughput | 696,136.44 examples/s |
| Batch 128 median | 0.049024 ms |
| Batch 128 throughput | 2,610,966.06 examples/s |
| Batch 128 peak allocation | 67.000 MiB |


## 7. Interpret rather than merely print

Median operator latency stayed near 0.046 ms from batch 1 through 32, so derived throughput rose from 21,656 to 696,136 examples/s. Batch 128 increased median to 0.049024 ms but still reached 2.61 million examples/s. Peak allocated memory grew from 64.023 to 67.000 MiB.

The table shows why throughput can improve dramatically while latency barely changes and memory rises. It does not include the time a request waits to join that batch, which may dominate an interactive SLO.

**Inspection rule:** Compare all axes at the same shape and precision. This is an operator workload, not a vLLM service benchmark.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Batching changed throughput, latency, and memory in different directions; no service queueing was modeled.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:12+00:00",
  "lesson": 24,
  "operator_workload": [
    {
      "batch": 1,
      "examples_per_second": 21656.27,
      "peak_allocated_mib": 64.023,
      "timing": {
        "median_ms": 0.046176,
        "p90_ms": 0.048736,
        "repeats": 20,
        "samples_ms": [
          0.0648,
          0.054496,
          0.048736,
          0.046016,
          0.04672,
          0.046752,
          0.04592,
          0.046144,
          0.04768,
          0.046496,
          0.045792,
          0.046688,
          0.046816,
          0.045568,
          0.045632,
          0.04544,
        

## 9. Make the bounded decision

> Choose a candidate against a service-level objective, not the single largest throughput number.

**Acceptance/rollback:** Declare workload distribution, warm-up, repetitions, synchronization, concurrency, percentile method, precision, and SLO before seeing the candidate.

**Failure analysis:** Comparing tokens/s at different output lengths or latency percentiles is unfair. Deriving service throughput from a single operator omits non-quantized layers and scheduling. Another trap is reporting only the best concurrency before OOM or rejection, without a safety margin and sustained-load duration.


## 10. Extend the evidence

Run a real serving workload with a frozen prompt/output-length distribution and arrival process. Record TTFT, inter-token latency, end-to-end p50/p95/p99, tokens/s, requests/s, queue depth, rejection, power, and peak/reserved memory for each concurrency level.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
